# Estadísticas de la base de 340 sintonizaciones (experimento E5)

En esta libreta se describe la base de parámetros de la Arquitectura I sintonizados mediante algoritmo genético para los 340 pacientes virtuales (archivo `SINTONIZACIONES.csv`, generado en `PID_Filtro_EMA.ipynb`).

**Procedimiento.**
1. Distribución del RMSE por sexo y prueba U de Mann–Whitney.
2. Estadística descriptiva de los parámetros, coeficiente de variación y número de parámetros en los límites del intervalo de búsqueda.
3. Correlaciones de Spearman entre las covariables del paciente y los parámetros sintonizados.

En el archivo, la columna `gamma` corresponde al factor de escalamiento denotado como ρ en la tesis. 

In [1]:
# =============================================================================
# CONFIGURACIÓN Y CARGA DE DATOS
# =============================================================================
import numpy as np
import pandas as pd
from scipy import stats

RUTA_SINTONIZACIONES = "SINTONIZACIONES.csv"
LIM_INF, LIM_SUP     = 1e-8, 3.0          # Intervalo de búsqueda del experimento E5
UMBRAL_RMSE          = 16.0               # Umbral de sintonización deficiente
PARAMETROS           = ["Kp", "Ki", "Kd", "gamma"]
COVARIABLES          = ["Edad", "Peso", "Altura"]

s = pd.read_csv(RUTA_SINTONIZACIONES)
print(f"[INFO] Pacientes: {len(s)}   hombres: {(s.Sexo == 'M').sum()}   mujeres: {(s.Sexo == 'F').sum()}")

[INFO] Pacientes: 340   hombres: 170   mujeres: 170


## Etapa 1: Desempeño por sexo

In [2]:
# =============================================================================
# RMSE POR SEXO Y CASOS CON SINTONIZACIÓN DEFICIENTE
# =============================================================================
print(s.groupby("Sexo")["RMSE"].describe().round(3))
u = stats.mannwhitneyu(s.loc[s.Sexo == "M", "RMSE"], s.loc[s.Sexo == "F", "RMSE"])
print(f"\nU de Mann-Whitney (hombres frente a mujeres): valor-p = {u.pvalue:.2e}")
print(f"RMSE total: {s.RMSE.mean():.2f} ± {s.RMSE.std():.2f}   "
      f"(mín. {s.RMSE.min():.2f}, máx. {s.RMSE.max():.2f})")
print("Casos con RMSE mayor que el umbral, por sexo:")
print((s.RMSE > UMBRAL_RMSE).groupby(s.Sexo).sum())

      count    mean    std    min    25%     50%     75%     max
Sexo                                                            
F     170.0  10.721  3.863  4.491  7.078  10.761  14.071  21.084
M     170.0  11.843  3.488  7.355  9.083   9.898  15.596  20.887

U de Mann-Whitney (hombres frente a mujeres): valor-p = 6.46e-04
RMSE total: 11.28 ± 3.72   (mín. 4.49, máx. 21.08)
Casos con RMSE mayor que el umbral, por sexo:
Sexo
F    14
M    36
Name: RMSE, dtype: int64


## Etapa 2: Distribución de los parámetros sintonizados

In [3]:
# =============================================================================
# ESTADÍSTICA DESCRIPTIVA DE LOS PARÁMETROS
# =============================================================================
filas = []
for c in PARAMETROS:
    v = s[c]
    filas.append(dict(Parametro=c,
                      Media=v.mean(), Desv_est=v.std(), Mediana=v.median(),
                      Q1=v.quantile(0.25), Q3=v.quantile(0.75),
                      Coef_variacion=v.std() / v.mean(),
                      En_limite_inferior=int((v <= 1e-7).sum()),
                      En_limite_superior=int((v >= LIM_SUP - 1e-3).sum())))
pd.DataFrame(filas).round(4)

,Parametro,Media,Desv_est,Mediana,Q1,Q3,Coef_variacion,En_limite_inferior,En_limite_superior
0,Kp,1.7761,0.7258,1.8258,1.3396,2.3383,0.4086,0,17
1,Ki,0.0057,0.0139,0.0021,0.0007,0.0065,2.4263,46,0
2,Kd,1.4474,0.7880,1.4815,0.8839,2.0238,0.5444,11,11
3,gamma,1.5319,0.7409,1.4838,1.0239,2.1151,0.4836,1,7


## Etapa 3: Relación entre covariables y parámetros

In [4]:
# =============================================================================
# CORRELACIONES DE SPEARMAN ENTRE COVARIABLES Y PARÁMETROS
# =============================================================================
filas = []
for c in PARAMETROS:
    for v in COVARIABLES + ["RMSE"]:
        r = stats.spearmanr(s[c], s[v])
        filas.append(dict(Parametro=c, Variable=v, rho_Spearman=r.statistic, Valor_p=r.pvalue))
correlaciones = pd.DataFrame(filas)
correlaciones.pivot(index="Variable", columns="Parametro", values="rho_Spearman").round(3)

Parametro,Kd,Ki,Kp,gamma
Variable,,,,
Altura,-0.017,0.315,0.172,0.141
Edad,0.026,-0.082,-0.043,-0.054
Peso,-0.043,0.153,0.095,0.097
RMSE,-0.105,0.039,-0.063,-0.016
